# 🚀 Colab A100 Dense 400M End-to-End Pipeline
**Arsitektur:** Dense 399.84M Parameter (32 Layers, Hidden 1024, 16 Q Heads, 4 KV Heads, SwiGLU 2880, RoPE, RMSNorm)  
**Target:** Pretraining ~2.1B Tokens + SFT + SimPO Preference Alignment  
**Optimasi Token/s:** NVMe Local I/O (working directory), Triton Inductor (`torch.compile`), TF32, Fused AdamW, Async Checkpoint Sync (Zero GPU Stalling).  
**Paths:** set `DRIVE_MOUNT` and `WORK_DIR` environment variables before running; this notebook stores no machine-specific absolute paths.


In [ ]:
# === 1. Pemeriksaan GPU & Driver ===
import torch, subprocess, os

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported  : {torch.cuda.is_bf16_supported()}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 Matmul     : AKTIF")
!nvidia-smi


In [ ]:
# === 2. Mount Google Drive ===
import os
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path(os.environ.get("DRIVE_MOUNT", "drive"))
if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "Transformer_400M_Colab"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Google Drive terhubung di: {DRIVE_ROOT}")


In [ ]:
# === 3. Konfigurasi Pipeline & Hyperparameter 400M ===
import os
from pathlib import Path

# --- Direktori Kerja Lokal (NVMe SSD Super Cepat) ---
WORK_DIR = Path(os.environ.get("WORK_DIR", "work"))
PROJECT_ROOT = WORK_DIR / "TUBES_1_400M"
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# --- Hyperparameter Pretrain ---
PRETRAIN_BATCH_SIZE = 16      # Micro-batch per step
PRETRAIN_GRAD_ACCUM = 4       # Akumulasi 4 micro-batch -> Efektif 64 sequence (65.536 token/step)
PRETRAIN_SEQ_LEN = 1024       # Konteks pretrain
PRETRAIN_LR = 3e-4            # Peak LR
WARMUP_STEPS = 500
MAX_PRETRAIN_STEPS = 32000    # ~2.1B Tokens
SAVE_INTERVAL = 1000          # Simpan checkpoint per 1000 step
EVAL_INTERVAL = 1000

# --- Performa & Kompilasi ---
USE_COMPILE = True            # Triton Inductor torch.compile (Fastest Token/s)
COMPILE_MODE = "max-autotune-no-cudagraphs"
ASYNC_SYNC_INTERVAL = 60      # Detik antar sinkronisasi checkpoint ke Drive di background
AUTO_DISCONNECT = True        # Putus runtime otomatis setelah pipeline selesai (Hemat CU)

print("Konfigurasi 400M berhasil dimuat!")


In [ ]:
# === 4. Ekstraksi Kode & Dataset ke NVMe SSD ===
import os, shutil, tarfile, time

t0 = time.time()

# 1. Ekstrak Kode 400M
code_tar = DRIVE_ROOT / "01_code_400m.tar"
if code_tar.exists():
    print(f"Mengekstrak kode dari {code_tar.name} ...")
    with tarfile.open(code_tar, "r:*") as tf:
        tf.extractall(PROJECT_ROOT)
    print("Kode 400M siap di:", PROJECT_ROOT)
else:
    print(f"Perhatian: {code_tar.name} tidak ditemukan di Drive. Memeriksa folder kode langsung...")

# 2. Ekstrak Dataset Pretrain Tokenized
tok_dir = DATA_DIR / "tokenized"
if not (tok_dir / "index.json").exists():
    tok_dir.mkdir(parents=True, exist_ok=True)
    for tar_name in ["03_tokenized_part1.tar", "04_tokenized_part2.tar"]:
        p = DRIVE_ROOT / tar_name
        if p.exists():
            print(f"Mengekstrak {tar_name} ke NVMe {tok_dir} ...")
            with tarfile.open(p, "r:*") as tf:
                tf.extractall(tok_dir)
            print(f"Selesai ekstrak {tar_name}")
else:
    print("Dataset tokenized sudah siap di NVMe:", tok_dir)

# 3. Ekstrak Dataset SFT & Preference (untuk Post-Training)
sft_pref_tar = DRIVE_ROOT / "02_dataset_sft_preference.tar"
if not (DATA_DIR / "sft" / "train.jsonl").exists() and sft_pref_tar.exists():
    print(f"Mengekstrak {sft_pref_tar.name} ...")
    with tarfile.open(sft_pref_tar, "r:*") as tf:
        tf.extractall(PROJECT_ROOT)
    print("Dataset SFT & Preference siap!")

print(f"Ekstraksi selesai dalam {time.time() - t0:.1f} detik!")


In [ ]:
# === 5. Daemon Sinkronisasi Checkpoint NVMe -> Drive (Background Async) ===
import shutil, threading, time
from pathlib import Path

DRIVE_CKPT_ROOT = DRIVE_ROOT / "checkpoints"
DRIVE_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

sync_stop_signal = False

def async_drive_sync_worker(local_root: Path, drive_root: Path, interval: int = 60):
    print(f"[Async Syncer] Daemon aktif (interval {interval}s). Menyinkronkan checkpoint ke Google Drive tanpa mengganggu GPU.")
    while not sync_stop_signal:
        time.sleep(interval)
        try:
            if not local_root.exists():
                continue
            for stage in ["pretrain", "sft", "preference"]:
                loc_stage = local_root / stage
                drv_stage = drive_root / stage
                if not loc_stage.exists():
                    continue
                drv_stage.mkdir(parents=True, exist_ok=True)
                # Sinkronkan log & metrics
                for f in loc_stage.glob("*.log"):
                    shutil.copy2(f, drv_stage / f.name)
                for f in loc_stage.glob("*.jsonl"):
                    shutil.copy2(f, drv_stage / f.name)
                # Sinkronkan folder checkpoint step_*
                for step_dir in loc_stage.glob("step_*"):
                    dst_step = drv_stage / step_dir.name
                    # Hanya sinkronkan jika checkpoint sudah selesai ditulis (ada training_state.pt)
                    if (step_dir / "training_state.pt").exists():
                        if not (dst_step / "training_state.pt").exists():
                            shutil.copytree(step_dir, dst_step, dirs_exist_ok=True)
                # Sinkronkan final jika ada
                if (loc_stage / "final").exists():
                    shutil.copytree(loc_stage / "final", drv_stage / "final", dirs_exist_ok=True)
        except Exception as e:
            pass
    print("[Async Syncer] Daemon berhenti.")

def start_sync_daemon():
    global sync_stop_signal
    sync_stop_signal = False
    thread = threading.Thread(target=async_drive_sync_worker, args=(CHECKPOINT_DIR, DRIVE_CKPT_ROOT, ASYNC_SYNC_INTERVAL), daemon=True)
    thread.start()
    return thread

def stop_sync_and_flush():
    global sync_stop_signal
    sync_stop_signal = True
    time.sleep(1)
    print("Melakukan sinkronisasi final penuh ke Google Drive ...")
    try:
        shutil.copytree(CHECKPOINT_DIR, DRIVE_CKPT_ROOT, dirs_exist_ok=True)
        print("Semua checkpoint tersimpan aman di Google Drive!")
    except Exception as e:
        print("Sinkronisasi final selesai:", e)

# Jalankan daemon
sync_thread = start_sync_daemon()


In [ ]:
# === 6. Helper: Deteksi Checkpoint Resume Terbaru ===
def get_best_checkpoint(stage: str):
    candidates = []
    for base in [CHECKPOINT_DIR / stage, DRIVE_CKPT_ROOT / stage]:
        if base.exists():
            for p in base.glob("step_*"):
                if (p / "pytorch_model.bin").exists():
                    try:
                        step_num = int(p.name.split("_")[1])
                        candidates.append((step_num, p))
                    except Exception:
                        pass
    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        best_step, best_p = candidates[0]
        # Salin ke lokal jika belum ada di lokal
        local_target = CHECKPOINT_DIR / stage / best_p.name
        if not local_target.exists():
            print(f"Menyalin checkpoint resume {best_p.name} dari Drive ke NVMe...")
            shutil.copytree(best_p, local_target, dirs_exist_ok=True)
        return best_step, local_target
    return None, None

best_pt_step, best_pt_path = get_best_checkpoint("pretrain")
print(f"Status Checkpoint Pretrain: step={best_pt_step}, path={best_pt_path}")


In [ ]:
# === 7. STAGE 1: Pretraining Dense 400M (Target ~2.1B Tokens) ===
import os, subprocess, sys

best_pt_step, best_pt_path = get_best_checkpoint("pretrain")

cmd_pretrain = [
    sys.executable, "-u", str(PROJECT_ROOT / "train" / "train_pretrain.py"),
    "--model-dir", str(PROJECT_ROOT / "model"),
    "--data-index", str(DATA_DIR / "tokenized" / "index.json"),
    "--output-dir", str(CHECKPOINT_DIR / "pretrain"),
    "--seq-len", str(PRETRAIN_SEQ_LEN),
    "--batch-size", str(PRETRAIN_BATCH_SIZE),
    "--grad-accum", str(PRETRAIN_GRAD_ACCUM),
    "--lr", str(PRETRAIN_LR),
    "--warmup-steps", str(WARMUP_STEPS),
    "--max-steps", str(MAX_PRETRAIN_STEPS),
    "--save-interval", str(SAVE_INTERVAL),
    "--eval-interval", str(EVAL_INTERVAL),
    "--dtype", "bf16",
    "--num-workers", "4",
]

if USE_COMPILE:
    cmd_pretrain += ["--compile", "--compile-mode", COMPILE_MODE]

if best_pt_path is not None:
    cmd_pretrain += ["--resume", str(best_pt_path)]
    print(f"Melanjutkan pretraining dari step {best_pt_step}!")
else:
    print("Memulai pretraining 400M dari step 0 (Fresh run)!")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TORCHINDUCTOR_CACHE_DIR"] = str(WORK_DIR / "triton_cache")

print("Menjalankan Pretraining:")
print(" ".join(cmd_pretrain))

proc = subprocess.Popen(cmd_pretrain, cwd=str(PROJECT_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Pretraining gagal dengan exit code {rc}")
print("Pretraining 400M Selesai Sukses!")


In [ ]:
# === 8. STAGE 2: SFT (Supervised Fine-Tuning) ===
best_pt_step, best_pt_path = get_best_checkpoint("pretrain")
base_ckpt = (CHECKPOINT_DIR / "pretrain" / "final") if (CHECKPOINT_DIR / "pretrain" / "final").exists() else best_pt_path

if base_ckpt is None:
    raise RuntimeError("Tidak ada base checkpoint pretrain untuk SFT!")

best_sft_step, best_sft_path = get_best_checkpoint("sft")

cmd_sft = [
    sys.executable, "-u", str(PROJECT_ROOT / "train" / "train_sft.py"),
    "--base-checkpoint", str(base_ckpt),
    "--train-file", str(DATA_DIR / "sft" / "train.jsonl"),
    "--val-file", str(DATA_DIR / "sft" / "val.jsonl"),
    "--tokenizer-dir", str(DATA_DIR / "tokenizer"),
    "--output-dir", str(CHECKPOINT_DIR / "sft"),
    "--max-seq-len", "2048",
    "--batch-size", "16",
    "--grad-accum", "2",
    "--lr", "5e-5",
    "--warmup-steps", "100",
    "--epochs", "1",
    "--eval-interval", "0",
    "--save-interval", "1500",
    "--dtype", "bf16",
    "--num-workers", "2",
    "--no-gradient-checkpointing",
]

if USE_COMPILE:
    cmd_sft.append("--compile")

if best_sft_path is not None:
    cmd_sft += ["--resume", str(best_sft_path)]
    print(f"Melanjutkan SFT dari step {best_sft_step}!")
else:
    print("Memulai SFT 400M dari base checkpoint pretrain!")

proc_sft = subprocess.Popen(cmd_sft, cwd=str(PROJECT_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_sft.stdout:
    print(line, end="", flush=True)

rc_sft = proc_sft.wait()
if rc_sft != 0:
    raise RuntimeError(f"SFT gagal dengan exit code {rc_sft}")
print("SFT 400M Selesai Sukses!")


In [ ]:
# === 9. STAGE 3: SimPO Preference Alignment ===
best_sft_step, best_sft_path = get_best_checkpoint("sft")
sft_base = (CHECKPOINT_DIR / "sft" / "final") if (CHECKPOINT_DIR / "sft" / "final").exists() else best_sft_path

if sft_base is None:
    raise RuntimeError("Tidak ada base checkpoint SFT untuk SimPO!")

best_pref_step, best_pref_path = get_best_checkpoint("preference")

cmd_pref = [
    sys.executable, "-u", str(PROJECT_ROOT / "train" / "train_preference.py"),
    "--base-checkpoint", str(sft_base),
    "--train-file", str(DATA_DIR / "preference" / "train.jsonl"),
    "--val-file", str(DATA_DIR / "preference" / "val.jsonl"),
    "--tokenizer-dir", str(DATA_DIR / "tokenizer"),
    "--output-dir", str(CHECKPOINT_DIR / "preference"),
    "--loss-type", "simpo",
    "--simpo-beta", "2.0",
    "--simpo-gamma", "1.4",
    "--max-seq-len", "1024",
    "--batch-size", "8",
    "--grad-accum", "4",
    "--lr", "1e-5",
    "--warmup-steps", "50",
    "--epochs", "1",
    "--eval-interval", "0",
    "--save-interval", "500",
    "--dtype", "bf16",
    "--num-workers", "2",
]

if USE_COMPILE:
    cmd_pref.append("--compile")

if best_pref_path is not None:
    cmd_pref += ["--resume", str(best_pref_path)]
    print(f"Melanjutkan SimPO dari step {best_pref_step}!")
else:
    print("Memulai SimPO Preference Alignment dari model SFT!")

proc_pref = subprocess.Popen(cmd_pref, cwd=str(PROJECT_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_pref.stdout:
    print(line, end="", flush=True)

rc_pref = proc_pref.wait()
if rc_pref != 0:
    raise RuntimeError(f"SimPO gagal dengan exit code {rc_pref}")
print("SimPO Preference Alignment Selesai Sukses!")


In [ ]:
# === 10. Final Sync & Auto Disconnect ===
stop_sync_and_flush()

print("
=======================================================")
print("🎉 SELURUH PIPELINE 400M SELESAI DENGAN SEMPURNA!")
print("Semua checkpoint pretrain, SFT, dan SimPO tersimpan di Google Drive.")
print("=======================================================
")

if AUTO_DISCONNECT:
    print("Menonaktifkan runtime Google Colab untuk menghemat saldo CU...")
    from google.colab import runtime
    runtime.unassign()
